# Predicting Electric Vehicle Population in Washington State Counties
This project examines a dataset summarizing a range of statstics for 271,114 elecrtric vehicles (EV) registered in Washington state. The dataset (`Electric_Vehicle_Population_Data.csv`) provides the following statistics for each vehicle:


* **VIN (1-10):** vehicle identification number
* **County:** name of county the vehicle is registered in
* **City:** name of city the vehicle is registered in
* **State:** all vehicles in this dataset are registered in Washington state
* **Postal Code:** postal code the vehicle is registered in
* **Model Year:** manufacturing year of the vehicle
* **Make:** manufacturer of the vehicle
* **Model:** model name of the vehicle
* **Electric Vehicle Type:** type of EV
* **Clean Alternative Fuel Vehicle (CAFV) Eligibility:** either elibile for CAFV or not
* **Electric Range:** total distance an EV can travel on a single full charge before the battery is depleted in miles
* **Legislative District:** WA is divided into 49 legislative districts
* **DOL Vehicle ID:** unique internal identification number assigned by WA State Departement of Licensing (DOL)
* **Vehicle Location:** coordiantes of vehicle location
* **Electric Utility:** name of utility company
* **2020 Census Tract**


---
## Feature Selection and Preprocessing
The dataset statistics are provided at the vehicle level, but our modeling task operates at the county-year level. Therefore, careful feature selection and aggregation are required:

###Drop:
* **VIN (1-10):** no predictive meaning
* **City:** redundant with county
* **State:** constant (always WA)
* **Postal Code:** redundant with county
* **DOL Vehicle ID:** no predictive meaning
* **Vehicle Location:** redundant with county
* **Legislative District:** counties overlap multiple districts
* **2020 Census Tract**

###Keep:
* **County:** defines aggregation unit
* **Model Year:** allows county-year prediction
* **Model:** use to compute number of unique models per county-year
* **Electric Utility:** useful for clustering interpretation
* **Make:** used to compute manufacturer diversity
* **Electric Vehicle Type:** used to compute proportions of BEVs and PHEVs
* **CAFV Eligibility:** used to compute policy related adoption metrics
* **Electric Range:** aggregated as an average measure of technology maturity


---
## Objective


This project analyzes electric vehicle (EV) registration data from various counties in Washington state to:
1. **Regression:** Predict the number of EVs registered in each county using regression models.
2. **Clustering:** Identify patterns of EV adoption by clustering counties into adoption-level groups.

The results from this project aim to help stakeholders to make data-drive decisions about placement of charging stations, grid load forecasting, and environmental impact assessment.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor

sns.set(style="whitegrid")

In [2]:
# Load dataset
df = pd.read_csv('Electric_Vehicle_Population_Data.csv')

In [3]:
print("\nData Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())
display(df.describe(include='all').T)


Data Shape: (271113, 16)

Missing Values:
 VIN (1-10)                                             0
County                                                10
City                                                  10
State                                                  0
Postal Code                                           10
Model Year                                             0
Make                                                   0
Model                                                  0
Electric Vehicle Type                                  0
Clean Alternative Fuel Vehicle (CAFV) Eligibility      0
Electric Range                                         7
Legislative District                                 659
DOL Vehicle ID                                         0
Vehicle Location                                      88
Electric Utility                                      10
2020 Census Tract                                     10
dtype: int64


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
VIN (1-10),271113,16671,7SAYGDEE7P,1153,NaN,NaN,NaN,NaN,NaN,NaN,NaN
County,271103,246,King,133815,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,271103,870,Seattle,42105,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,271113,51,WA,270454,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Postal Code,271103.0,NaN,NaN,NaN,98176.923405,2576.510222,1030.0,98052.0,98133.0,98382.0,99577.0
Model Year,271113.0,NaN,NaN,NaN,2021.995068,3.05673,1999.0,2021.0,2023.0,2024.0,2026.0
Make,271113,47,TESLA,110537,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Model,271113,184,MODEL Y,57324,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Electric Vehicle Type,271113,2,Battery Electric Vehicle (BEV),216316,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Clean Alternative Fuel Vehicle (CAFV) Eligibility,271113,3,Eligibility unknown as battery range has not b...,170815,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Feature Selection and Preprocessing
cols_to_drop = [
    "VIN (1-10)",
    "City",
    "State",
    "Postal Code",
    "DOL Vehicle ID",
    "Vehicle Location",
    "Legislative District",
    "2020 Census Tract"
]
df = df.drop(columns=cols_to_drop, errors="ignore")

# Since there are only 27/271103 rows with missing values, removing these rows will not
# introduce bias or meaningfully affect county-level EV counts or feature distributions.
df = df.dropna(subset=["County", "Electric Range", "Electric Utility"])

# Count how many times each manufacture appears, and replace manufacturers that appear
# fewer than 1000 times with "Other". This reduces noise and improves generalization
make_counts = df["Make"].value_counts()
rare_makes = make_counts[make_counts < 1000].index
df["Make"] = df["Make"].replace(rare_makes, "Other")

# Display first 5 rows after selection and processing
display(df.head())

,County,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Electric Utility
0,Kitsap,2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215.0,PUGET SOUND ENERGY INC
1,Kitsap,2011,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,73.0,PUGET SOUND ENERGY INC
2,King,2011,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,73.0,PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA)
3,Yakima,2020,CHEVROLET,BOLT EV,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,259.0,PACIFICORP
4,King,2020,KIA,NIRO,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,26.0,CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA)


In [5]:
# Aggregate dataset by county-year
county_year_df = df.groupby(["County", "Model Year"]).agg(
    ev_count=("County", "size"),
    avg_range=("Electric Range", "mean"),
    # avg_msrp=("Base MSRP", "mean"),
    tesla_share=("Make", lambda x: (x == "TESLA").mean()),
    unique_makes=("Make", "nunique"),
    bev_share=("Electric Vehicle Type", lambda x: (x == "Battery Electric Vehicle (BEV)").mean()),
    cafv_share=("Clean Alternative Fuel Vehicle (CAFV) Eligibility",
                lambda x: (x == "Eligible").mean())
).reset_index()

# One-hot enocde County
county_year_df = pd.get_dummies(
    county_year_df,
    columns=["County"],
    drop_first=True
)

county_year_df["log_ev_count"] = np.log1p(county_year_df["ev_count"])

display(county_year_df.head())
print("County-year shape:", county_year_df.shape)
print(county_year_df.isnull().sum())

,Model Year,ev_count,avg_range,tesla_share,unique_makes,bev_share,cafv_share,County_Adams,County_Alachua,County_Alameda,...,County_Weber,County_Weld,County_Whatcom,County_Whitman,County_Williamson,County_Yakima,County_Yolo,County_York,County_Yuba,log_ev_count
0,2023,1,0.0,1.0,1,1.000000,0.0,False,False,False,...,False,False,False,False,False,False,False,False,False,0.693147
1,2011,2,73.0,0.0,1,1.000000,0.0,True,False,False,...,False,False,False,False,False,False,False,False,False,1.098612
2,2012,1,73.0,0.0,1,1.000000,0.0,True,False,False,...,False,False,False,False,False,False,False,False,False,0.693147
3,2013,1,19.0,0.0,1,0.000000,0.0,True,False,False,...,False,False,False,False,False,False,False,False,False,0.693147
4,2014,3,46.0,0.0,3,0.333333,0.0,True,False,False,...,False,False,False,False,False,False,False,False,False,1.386294


County-year shape: (1057, 253)
Model Year       0
ev_count         0
avg_range        0
tesla_share      0
unique_makes     0
                ..
County_Yakima    0
County_Yolo      0
County_York      0
County_Yuba      0
log_ev_count     0
Length: 253, dtype: int64


# Supervised Regresssion Modeling
To address skewness and exponential growth in EV registrations, the regression model predicts the logarithm of EV counts, allowing coefficients to be interpreted as percentage changes.

In [6]:
# Defining features
X = county_year_df.drop(columns=["ev_count", "log_ev_count"])
y = county_year_df["log_ev_count"]

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (845, 251)
X_test shape: (212, 251)


In [7]:
# Baseline model for mean EV counts
# Essentially ignores all features and serves as the minimum "benchmark" to be
# compared with models with features.
y_pred_base = np.full_like(y_test, y_train.mean(), dtype=float)

# Compute evaluation metrics
baseline_mse = mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2  = r2_score(y_test, y_pred_base)

print("Baseline Model:")
print("Mean Squared Error:", baseline_mse)
print("Mean Absolute Error:", baseline_mae)
print("R-squared:", baseline_r2)

Baseline Model:
Mean Squared Error: 4.764445973153985
Mean Absolute Error: 1.8583912471453854
R-squared: -1.1266502764950559e-05


In [8]:
# Linear Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

# Compute evaluation metrics
lr_mse = mean_squared_error(y_test, y_pred_lr)
lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_r2 = r2_score(y_test, y_pred_lr)

print("Linear Regression (Log Target):")
print("Mean Squared Error:", lr_mse)
print("Mean Absolute Error:", lr_mae)
print("R-squared:", lr_r2)

Linear Regression (Log Target):
Mean Squared Error: 0.31734780003112273
Mean Absolute Error: 0.3583333978950404
R-squared: 0.9333917569389694


In [9]:
# Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

y_pred_ridge = ridge.predict(X_test_scaled)

# Compute evaluation metrics
ridge_mse = mean_squared_error(y_test, y_pred_ridge)
ridge_mae = mean_absolute_error(y_test, y_pred_ridge)
ridge_r2 = r2_score(y_test, y_pred_ridge)

print("Ridge Regression:")
print("Mean Squared Error:", ridge_mse)
print("Mean Absolute Error:", ridge_mae)
print("R-squared:", ridge_r2)

Ridge Regression:
Mean Squared Error: 0.35714173313556274
Mean Absolute Error: 0.4118306909820453
R-squared: 0.9250393941108199


In [10]:
# Comparison table (for readability)
results = pd.DataFrame([
    {"Model": "Baseline (Mean)", "MSE": baseline_mse, "MAE": baseline_mae, "R2": baseline_r2},
    {"Model": "Linear Regression", "MSE": lr_mse, "MAE": lr_mae, "R2": lr_r2},
    {"Model": "Ridge Regression", "MSE": ridge_mse, "MAE": ridge_mae, "R2": ridge_r2},
])

display(results)

,Model,MSE,MAE,R2
0,Baseline (Mean),4.764446,1.858391,-0.000011
1,Linear Regression,0.317348,0.358333,0.933392
2,Ridge Regression,0.357142,0.411831,0.925039


# Decision Tree Regression

In [13]:
# Decision Tree Regressor
dt = DecisionTreeRegressor(
    max_depth=6,        # controls overfitting
    min_samples_leaf=10,
    random_state=42
)

dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# Evaluation metrics
dt_mse = mean_squared_error(y_test, y_pred_dt)
dt_mae = mean_absolute_error(y_test, y_pred_dt)
dt_r2  = r2_score(y_test, y_pred_dt)

In [18]:
results = pd.DataFrame([
    {"Model": "Baseline (Mean)", "MSE": baseline_mse, "MAE": baseline_mae, "R2": baseline_r2},
    {"Model": "Linear Regression", "MSE": lr_mse, "MAE": lr_mae, "R2": lr_r2},
    {"Model": "Ridge Regression", "MSE": ridge_mse, "MAE": ridge_mae, "R2": ridge_r2},
    {"Model": "Decision Tree", "MSE": dt_mse, "MAE": dt_mae, "R2": dt_r2},
])

display(results)

,Model,MSE,MAE,R2
0,Baseline (Mean),4.764446,1.858391,-0.000011
1,Linear Regression,0.317348,0.358333,0.933392
2,Ridge Regression,0.357142,0.411831,0.925039
3,Decision Tree,0.413347,0.394941,0.913242
